# wav2vec2 fine-tuning on CORAAL (DCB)
Fine-tunes `facebook/wav2vec2-large-960h-lv60-self` on CORAAL DCB, loading the data exactly as `Whisper_CORAAL_finetune.ipynb` does
(same 2,145 train / 239 eval split), and saves zero-shot and fine-tuned predictions for PER/WER scoring.

Runtime → Change runtime type → **GPU** (T4 works; A100/L4 is faster).

## Prepare Environment

In [ ]:
!pip install --upgrade pip
!pip install --upgrade "datasets[audio]" transformers accelerate evaluate jiwer

## Load Dataset

Same loading code as `Whisper_CORAAL_finetune.ipynb`, so `train_test_split(test_size=0.1, seed=42)` gives the identical 2,145 / 239 split.

In [ ]:
!git clone https://github.com/stanford-policylab/asr-disparities.git

In [ ]:
!ls asr-disparities/input/

In [ ]:
from glob import glob
# coraal_audio = sorted(glob("asr-disparities/input/CORAAL_audio/*/*.mp3"))
coraal_audio = sorted(glob("asr-disparities/input/CORAAL_audio/dcb/*.mp3"))

In [ ]:
import pandas as pd
gt_csv = "asr-disparities/input/CORAAL_transcripts.csv" #csv with ground truth transcripts
gt_df = pd.read_csv(gt_csv)
gt_df = gt_df.sort_values(by='segment_filename')
gt_df.head()

Make sure to only get audio files with their transcript present in the test set

In [ ]:
base_audio_name =[item.split("/")[-1].split(".")[0] for item in coraal_audio]
base_transcript = [item.replace(".wav","") for item in gt_df['segment_filename'].tolist()]
use_set = sorted(list(set(base_audio_name) & set(base_transcript)))

coraal_audio_set = []
for item in use_set:
    get_file = glob(f"asr-disparities/input/CORAAL_audio/*/{item}.mp3")
    coraal_audio_set.append(get_file[0])

In [ ]:
import os
import re

gt_trans = []
file_names = []

# Create a dictionary for efficient lookup: base_filename_without_ext -> content
gt_content_map = {
    os.path.splitext(row['segment_filename'])[0]: row['content']
    for index, row in gt_df.iterrows()
}

for audio_path in coraal_audio_set:
    base_audio_name = os.path.splitext(os.path.basename(audio_path))[0]
    file_names.append(os.path.basename(audio_path))
    transcript = gt_content_map.get(base_audio_name, '')
    # Remove any annotation notes in angle brackets (e.g., <ts>, <noise>)
    transcript = re.sub(r'<.*?>', '', transcript).strip()
    gt_trans.append(transcript)

print(gt_trans[0])

In [ ]:
df_audio_files = pd.DataFrame({'file_name': file_names})
df_audio_files["transcription"] = gt_trans
print(f"DataFrame created with {len(file_names)} audio file paths.")
csv_file_path = "asr-disparities/input/CORAAL_audio/dcb/metadata.csv"
df_audio_files.to_csv(csv_file_path, index=False)

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("asr-disparities/input/CORAAL_audio/dcb")
dataset = dataset['train']
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(train_dataset)
print(eval_dataset)

In [ ]:
# Optional check: upload whisper_baseline.csv to /content to confirm the eval split matches the Whisper run
if os.path.exists("whisper_baseline.csv"):
    assert pd.read_csv("whisper_baseline.csv").reference.fillna("").tolist() == eval_dataset["transcription"]
    print("eval split matches whisper_baseline.csv")

In [ ]:
import numpy as np

def target_text(t):
    # CTC targets use the model's vocab (A-Z, apostrophe, word delimiter)
    t = re.sub(r"/RD-[^/]*/|/\?+/", " ", t)          # drop redaction tags and unintelligible marks
    t = t.replace("/", " ").replace("-", " ").upper()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z' ]", " ", t)).strip()

# decode every clip once (16 kHz) so training doesn't re-decode mp3s each epoch
train_audio = [np.asarray(ex["audio"]["array"], dtype=np.float32) for ex in train_dataset]
train_text = [target_text(t) for t in train_dataset["transcription"]]
train_dur = [len(a) / 16000 for a in train_audio]
eval_audio = [np.asarray(ex["audio"]["array"], dtype=np.float32) for ex in eval_dataset]
references = eval_dataset["transcription"]

keep = [i for i, t in enumerate(train_text) if t]
print(len(train_audio), "train clips,", len(keep), "with non-empty targets,", round(sum(train_dur) / 3600, 2), "hours")

## Load Model

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

MODEL_CHECKPOINT = "facebook/wav2vec2-large-960h-lv60-self"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = Wav2Vec2Processor.from_pretrained(MODEL_CHECKPOINT)

def load_model():
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_CHECKPOINT, ctc_loss_reduction="mean", ctc_zero_infinity=True, layerdrop=0.0)
    # masked_spec_embed is missing from this checkpoint and loads uninitialized (NaN), which turns every
    # SpecAugment-masked frame into NaN during training; re-initialize it the way the model's _init_weights does
    model.wav2vec2.masked_spec_embed.data.uniform_()
    return model.to(device)

model = load_model()

## Inference

In [ ]:
@torch.no_grad()
def transcribe(model, audios):
    model.eval(); preds = []
    for a in audios:
        x = processor(a, sampling_rate=16000, return_tensors="pt").input_values.to(device)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            logits = model(x).logits
        preds.append(processor.batch_decode(logits.argmax(-1))[0])
    return preds

zero_shot_predictions = transcribe(model, eval_audio)
pd.DataFrame({"reference": references, "zero_shot_prediction": zero_shot_predictions}).to_csv("wav2vec2_baseline.csv", index=False)

## Train

In [ ]:
import math, random, time
from transformers import get_linear_schedule_with_warmup

EPOCHS, LR, MAX_BATCH_SECS, ACCUM = 5, 3e-5, 60, 2
torch.manual_seed(42); random.seed(42)

def make_batches():
    # bucket by duration so each batch holds <= MAX_BATCH_SECS of audio
    idx = sorted(keep, key=lambda i: train_dur[i] + random.random())
    out, cur, secs = [], [], 0
    for i in idx:
        if cur and secs + train_dur[i] > MAX_BATCH_SECS:
            out.append(cur); cur, secs = [], 0
        cur.append(i); secs += train_dur[i]
    out.append(cur); random.shuffle(out)
    return out

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
total_steps = EPOCHS * math.ceil(len(make_batches()) / ACCUM)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=device == "cuda")

model.train()
for epoch in range(1, EPOCHS + 1):
    t0, losses = time.time(), []
    batches = make_batches()
    for b, ids in enumerate(batches):
        feats = processor([train_audio[i] for i in ids], sampling_rate=16000, padding=True, return_tensors="pt")
        lab = processor.tokenizer([train_text[i] for i in ids], padding=True, return_tensors="pt")
        labels = lab.input_ids.masked_fill(lab.attention_mask.ne(1), -100)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            loss = model(feats.input_values.to(device), attention_mask=feats.attention_mask.to(device), labels=labels.to(device)).loss
        if not torch.isfinite(loss):
            print(f"skipping non-finite batch {b}"); opt.zero_grad(); continue
        scaler.scale(loss / ACCUM).backward(); losses.append(loss.item())
        if (b + 1) % ACCUM == 0 or b == len(batches) - 1:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        if b % 100 == 0:
            print(f"epoch {epoch} batch {b}/{len(batches)} loss {np.mean(losses[-100:]):.3f} lr {sched.get_last_lr()[0]:.2e} {time.time()-t0:.0f}s")
    print(f"EPOCH {epoch}: mean loss {np.mean(losses):.3f}, {(time.time()-t0)/60:.1f} min")
    model.train()

In [ ]:
model.save_pretrained("./wav2vec2-large-fine-tune")
processor.save_pretrained("./wav2vec2-large-fine-tune")

finetuned_predictions = transcribe(model, eval_audio)
finetuned_df = pd.DataFrame({"reference": references, "finetuned_prediction": finetuned_predictions})
finetuned_df.to_csv("wav2vec2_finetuned.csv", index=False)
finetuned_df.head()

## Evaluate

In [ ]:
!pip install -q jiwer
import jiwer
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]", " ", str(s).lower().replace("-", " "))).strip()
refs = [norm(r) for r in references]
print("Zero-shot WER (normalized):", jiwer.wer(refs, [norm(p) for p in zero_shot_predictions]))
print("Fine-tuned WER (normalized):", jiwer.wer(refs, [norm(p) for p in finetuned_predictions]))

## Save Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/asr_project/"
!cp "/content/wav2vec2_baseline.csv" "/content/wav2vec2_finetuned.csv" "/content/drive/MyDrive/asr_project/"
!cp -r "/content/wav2vec2-large-fine-tune" "/content/drive/MyDrive/asr_project/"